In [0]:
%run /Workspace/Users/senoom222@gmail.com/databricks-code-repos-senthil/Databricks_workout_2025/Calling_1_wb_to_2_wb_using_util_run/Generic_Specific_Frame

In [0]:
dbutils.widgets.text("Catalog","")
CATALOG = dbutils.widgets.get("Catalog").strip()
dbutils.widgets.text("Schema","")
SCHEMA = dbutils.widgets.get("Schema").strip()

In [0]:
%python
import json

child_output = dbutils.notebook.run("/Workspace/Users/senoom222@gmail.com/databricks-code-repos-senthil/Databricks_workout_2025/Calling_1_wb_to_2_wb_using_util_run/config",120,{"Catalog":CATALOG,"Schema":SCHEMA})

child_dict = json.loads(child_output)

CATALOG = child_dict["Catalog"]
SCHEMA = child_dict["Schema"]
SRC = child_dict["Source"]
BRONZE = child_dict["Bronze"]
SILVER = child_dict["Silver"]
GOLD = child_dict["Gold"]
SILVERTBL = child_dict["Silver Table"]
GOLDTBL = child_dict["Gold Table"]

SILVERTBL = SILVERTBL.split("/")[-1]

print("Returned Source Location : ",BRONZE)
print("Returned Target Location : ",SILVER)
print("Returned Target Table : ",SILVERTBL)


In [0]:
from pyspark.sql import functions as func

staff = spark.read.format("delta").load(f"{BRONZE}/logistics_staff")
geotag = spark.read.format("delta").load(f"{BRONZE}/logistics_geo")
shipment = spark.read.format("delta").load(f"{BRONZE}/logistics_shipment")

In [0]:
silver_staff = standardize_staff(staff)

silver_geotag=scrub_geotag(geotag).distinct()

silver_shipment = (shipment.where("shipment_weight_kg > 0")\
                    .transform(standardize_shipments)\
                    .transform(enrich_shipments)\
                    .transform(split_columns))


In [0]:
deltawrite(silver_staff,f"{SILVER}/logistics_silver_staff",mode = "overwrite",format="delta")
deltawrite(silver_geotag,f"{SILVER}/logistics_silver_geotag",mode = "overwrite",format="delta")
deltawrite(silver_shipment,f"{SILVER}/logistics_silver_shipment",mode = "overwrite",format="delta")

write_table(silver_staff,f"{SILVERTBL}.silver_staff", mode="overwrite")
write_table(silver_geotag,f"{SILVERTBL}.silver_geotag", mode="overwrite")
write_table(silver_shipment,f"{SILVERTBL}.silver_shipments", mode="overwrite")